## Step 1: Date Column Ko Proper Date Type Banana
- parse_dates=['col_name'] → read_csv() ke andar hi diya toh column automatically date type ban jaata hai
- Isse pehle alag se pd.to_datetime() call karna padta tha, ab shortcut mil gaya

In [ ]:
import pandas as pd
df = pd.read_csv("datasets/weather_data.csv", parse_dates=['day'])
df

## Step 2: Date Ko Index Banana
- set_index() → wahi purana concept, ab date proper type mein hai isliye time-based operations (jaise interpolate method="time") kaam karenge

In [ ]:
df.set_index('day',inplace=True)
df

## fillna() — Missing Values Ko Fixed Value Se Bharna
- fillna(0) → SABHI NaN ko 0 se bhar do (blanket rule, har column ke liye same)
- fillna({dict}) → column-specific rule (behtar tarika, kyunki har column ka context alag hota hai)
  - Example: temperature/windspeed ke NaN ko 0, lekin event (text) ke NaN ko "no event"

In [ ]:
new_df = df.fillna(0)
new_df

In [ ]:
new_df = df.fillna({
    'temperature' : 0,
    'windspeed' : 0,
    'event' : "no event"
})
new_df

## ffill() aur bfill() — Aas-Paas Ki Value Se Bharna
- ffill() → upar (pichli) wali value se NaN bharo (forward fill)
- bfill() → neeche (agli) wali value se NaN bharo (backward fill)
- axis="columns" → row ke andar LEFT-RIGHT fill karo (columns ke beech), kam common use case
  (default axis="index" hota hai, matlab row-wise/upar-neeche)

In [ ]:
new_df = df.ffill()
new_df

In [ ]:
new_df = df.bfill()
new_df

In [ ]:
new_df = df.bfill(axis="columns")
new_df

In [ ]:
new_df = df.ffill(axis="columns")
new_df

In [ ]:
## limit Parameter — Kitni Baar Tak Fill Karna Hai
- ffill(limit=1) → sirf 1 consecutive NaN tak fill karo, uske baad NaN hi rehne do
- Real use: chhote gaps (1-2 din missing) fill karna sahi hai, bade gaps (hafta bhar missing) fill karna misleading hoga

In [ ]:
new_df = df.ffill(limit=1)
new_df


## interpolate() — Mathematically Estimate Karke Bharna
- method="time" → date-index ke actual gaps ke hisaab se accurate estimate (index DatetimeIndex hona chahiye)
- IMPORTANT: text (object dtype) columns pe interpolate karna warning deta hai
- FIX: pehle select_dtypes(include='number') se sirf numeric columns nikaal lo, phir interpolate karo
  (numeric_only=True parameter is version mein reliably kaam nahi karta — select_dtypes zyada reliable hai)

In [ ]:
new_df = df.select_dtypes(include='number').interpolate(method="time")
new_df

## dropna() — Missing Data Wali Rows Ko Hata Dena
- dropna() → agar row mein EK BHI NaN hai, poori row drop (most aggressive/strict)
- dropna(how='all') → sirf tab drop karo jab SAARI values NaN hon (least aggressive)
- dropna(thresh=n) → row mein kam se kam n non-NaN values honi chahiye, warna drop (balanced approach)

In [ ]:
new_df = df.dropna()
new_df

In [ ]:
new_df = df.dropna(how='all')
new_df

In [ ]:
new_df = df.dropna(thresh=2)
new_df

## reindex() — Missing Dates Ko Explicitly Add Karna
- pd.date_range(start, end) → dono dates ke beech ki COMPLETE date list banata hai (koi din skip nahi)
- reindex(idx) → DataFrame ko is complete list ke hisaab se force karta hai
- Jo dates original data mein nahi thi, unke columns automatically NaN ban jaate hain
- Real use: time-series data mein missing dates ko chhupane ki bajaye explicitly NaN dikhana, taaki properly handle ho sakein

In [ ]:
dt = pd.date_range("2017-01-01", "2017-01-11")
idx = pd.DatetimeIndex(dt)

df = df.reindex(idx)
df